In [1]:
from langchain_community.document_loaders import TextLoader, DataFrameLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


/home/anurag/book-recommender/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. Load your API key
load_dotenv()
# In your setup code
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-exp",    
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.0
)
# AND CHANGE YOUR EMBEDDINGS TO:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

In [3]:
import pandas as pd 

books=pd.read_csv("books_cleaned.csv")

books["tagged_description"].to_csv("tagged_descriptions.text",
                                    sep="\n" , index=False, header=False)

In [4]:
# 1. Ensure TextLoader is imported (Fixes the NameError from before)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_data = TextLoader("tagged_descriptions.text").load()

# 2. Set chunk_size to 1000 (roughly the length of a book summary)
# and chunk_overlap to 100 (keeps context between chunks)
text_splitter = CharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100, 
    separator="\n"
)

# 3. This will now run without the ValueError
documents = text_splitter.split_documents(raw_data)



Created a chunk of size 1170, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1090, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1269, which is longer than the specified 1000
Created a chunk of size 2012, which is longer than the specified 1000
Created a chunk of size 1227, which is longer than the specified 1000
Created a chunk of size 1186, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1193, which is longer than the specified 1000
Created a chunk of size 1059, which is longer than the specified 1000
Created a chunk of size 1272, which is longer than the specified 1000
Created a chunk of size 1637, which is longer than the specified 1000
Created a chunk of size 1134, which is longer than the specified 1000
Created a chunk of s

Created a chunk of size 2014, which is longer than the specified 1000
Created a chunk of size 1288, which is longer than the specified 1000
Created a chunk of size 1233, which is longer than the specified 1000
Created a chunk of size 1244, which is longer than the specified 1000
Created a chunk of size 1128, which is longer than the specified 1000
Created a chunk of size 1075, which is longer than the specified 1000
Created a chunk of size 1320, which is longer than the specified 1000
Created a chunk of size 2836, which is longer than the specified 1000
Created a chunk of size 1143, which is longer than the specified 1000
Created a chunk of size 1006, which is longer than the specified 1000
Created a chunk of size 2512, which is longer than the specified 1000
Created a chunk of size 1424, which is longer than the specified 1000
Created a chunk of size 1085, which is longer than the specified 1000
Created a chunk of size 1146, which is longer than the specified 1000
Created a chunk of s

In [7]:
import os
import shutil
import time
import chromadb
from langchain_chroma import Chroma

# 1. NEW NAME to avoid the OS lock
db_path = "./book_db_final_version" 

# 2. Force delete if it somehow exists
if os.path.exists(db_path):
    shutil.rmtree(db_path)

# 3. Create the first batch
print("Starting initial ingestion...")
db_books = Chroma.from_documents(
    documents=documents[:40], 
    embedding=embeddings, 
    persist_directory=db_path
)

# 4. Batch loop
batch_size = 40
for i in range(batch_size, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    db_books.add_documents(batch)
    print(f"✅ Progress: {i + len(batch)} books stored.")
    time.sleep(10) # Essential for Gemini Free Tier

Starting initial ingestion...
✅ Progress: 80 books stored.
✅ Progress: 120 books stored.
✅ Progress: 160 books stored.
✅ Progress: 200 books stored.
✅ Progress: 240 books stored.
✅ Progress: 280 books stored.
✅ Progress: 320 books stored.
✅ Progress: 360 books stored.
✅ Progress: 400 books stored.
✅ Progress: 440 books stored.
✅ Progress: 480 books stored.
✅ Progress: 520 books stored.
✅ Progress: 560 books stored.
✅ Progress: 600 books stored.
✅ Progress: 640 books stored.
✅ Progress: 680 books stored.
✅ Progress: 720 books stored.
✅ Progress: 760 books stored.
✅ Progress: 800 books stored.
✅ Progress: 840 books stored.
✅ Progress: 880 books stored.
✅ Progress: 920 books stored.
✅ Progress: 960 books stored.
✅ Progress: 1000 books stored.
✅ Progress: 1040 books stored.
✅ Progress: 1080 books stored.
✅ Progress: 1120 books stored.
✅ Progress: 1160 books stored.
✅ Progress: 1200 books stored.
✅ Progress: 1240 books stored.
✅ Progress: 1280 books stored.
✅ Progress: 1320 books stored.
✅ 

In [5]:
import os
from langchain_chroma import Chroma

# Use the exact same path and embedding function
db_path = "book_db_final_version"

if os.path.exists(db_path):
    # This only LOADS the database. It does NOT create it or call the API for new embeddings.
    db_books = Chroma(
        persist_directory=db_path,
        embedding_function=embeddings
    )
    print(f"✅ Database loaded successfully with {len(db_books.get()['ids'])} books.")
else:
    print("❌ Error: Database folder not found. Did you delete it?")

✅ Database loaded successfully with 2941 books.


In [6]:
query = "A book about Roman history"
results = db_books.similarity_search(query, k=10)
print("results: ", results)


results:  [Document(id='e06f4902-c1c4-444a-a0e9-331c426f7584', metadata={'source': 'tagged_descriptions.text'}, page_content="9780380710829 | The lives of ancient Rome's men--general Gaius Marius and his rival Lucius Cornelius Sulla--unfold amid Republican Rome's struggle in a world of treachery and barbarism\n9780380710843 | The fourth novel of the Masters of Rome series focuses on the women in the life of the Roman emperor Gaius Julius Caesar at the height of his power"), Document(id='ee614b88-ab5d-4d91-b49c-96f0e7533fd7', metadata={'source': 'tagged_descriptions.text'}, page_content="9780760768952 | Among the most durable and engaging texts in world literature, Julius Caesar's Conquest of Gaul tells how he and his legions conquered much of modern France in less than a decade (58-51 BCE), despite determined resistance. Perhaps the most famous Roman ever, Gaius Julius Caesar created a legacy which has resonated, for good or ill, throughout Western culture. Architect of an imperial sys

In [7]:
# .replace('"', '') removes any double quotes found in that first segment
isbn_str = results[0].page_content.split(" ")[0].replace('"', '').strip()
books[books["isbn13"] == int(isbn_str)]
# print(results[0].metadata)

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
1753,9780380710829,038071082X,The Grass Crown,Colleen McCullough,Fiction,http://books.google.com/books/content?id=Kt3gN...,1992.0,4.29,1104.0,9397.0,The Grass Crown,9780380710829 | The lives of ancient Rome's me...


In [8]:
def retrieve_symmentaic_recommendations(
        query: str,
        top_k: int = 10
    ,
)-> pd.DataFrame:
    recs=db_books.similarity_search(query, k=top_k)
    book_list=[]
    for i in range(len(recs)):
        isbn_str = recs[i].page_content.split(" ")[0].replace('"', '').strip()
        book_info=books[books["isbn13"] == int(isbn_str)]
        book_list.append(book_info)
#    return pd.concat(book_list).reset_index(drop=True)
    return pd.concat(book_list).reset_index(drop=True)
    

In [9]:
results_df=retrieve_symmentaic_recommendations("A book about a cooking",top_k=10)
results_df

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780385607018,0385607016,The French Kitchen,Joanne Harris;Fran Warde,"Cooking, French",http://books.google.com/books/content?id=YYLqA...,2003.0,4.09,256.0,35.0,The French Kitchen: A Cookbook,9780385607018 | Joanne Harris's bestselling no...
1,9780060277406,0060277408,The Secret Garden Cookbook,Amy Cotler,Juvenile Nonfiction,http://books.google.com/books/content?id=c7E_H...,1999.0,4.28,128.0,142.0,The Secret Garden Cookbook: Recipes Inspired b...,9780060277406 | Frances Hodgson Burnett's The ...
2,9780446670784,0446670782,Curries Without Worries,Sudha Koul,Cooking,http://books.google.com/books/content?id=G4qUH...,1996.0,3.98,160.0,35.0,Curries Without Worries,9780446670784 | A thorough but accessible guid...
3,9780141010373,0141010371,Jamie's Kitchen,Jamie Oliver,Cookery,http://books.google.com/books/content?id=WG3SN...,2004.0,3.89,336.0,4012.0,Jamie's Kitchen,9780141010373 | The No 1 bestseller I love thi...
4,9781400051731,1400051738,The Ruby Ring,Diane Haeger,Fiction,http://books.google.com/books/content?id=uupUD...,2005.0,3.69,371.0,1050.0,The Ruby Ring: A Novel,9781400051731 | Seeking refuge in a convent fo...
5,9781401301958,1401301959,Jamie's Italy,Jamie Oliver,Cooking,http://books.google.com/books/content?id=e1sYA...,2006.0,4.00,320.0,7463.0,Jamie's Italy,9781401301958 | Bestselling author Jamie Olive...
6,9780811216371,0811216373,October Light,John Gardner,Fiction,http://books.google.com/books/content?id=wSDZU...,2005.0,3.88,399.0,823.0,October Light,9780811216371 | Living in her older brother's ...
7,9781904920359,1904920357,50 Great Curries of India 10th Anniversary Ed.,Camellia Panjabi,Cooking,http://books.google.com/books/content?id=h9JfS...,2005.0,4.17,224.0,197.0,50 Great Curries of India 10th Anniversary Ed.,9781904920359 | Collects various dishes from a...
8,9780771014178,0771014171,The Jane Austen Cookbook,Maggie Black;Deirdre Le Faye,Cooking,http://books.google.com/books/content?id=vtgBA...,2002.0,3.91,128.0,265.0,The Jane Austen Cookbook,9780771014178 | Jane Austen wrote her novels i...
9,9781400040308,1400040302,A Madman Dreams of Turing Machines,Janna Levin,Fiction,http://books.google.com/books/content?id=EwQi6...,2006.0,3.68,230.0,1098.0,A Madman Dreams of Turing Machines,"9781400040308 | In a saga of genius, madness, ..."


In [11]:
books['categories'].value_counts().reset_index().query('count>50')

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Religion,117
6,Philosophy,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [10]:
category_mapping = {'Fiction': "Fiction",
'Juvenile Fiction': "Children's Fiction",
'Biography & Autobiography': "Nonfiction",
'History': "Nonfiction",
'Literary Criticism': "Nonfiction",
'Philosophy': "Nonfiction",
'Religion': "Nonfiction",
'Comics & Graphic Novels': "Fiction",
'Drama': "Fiction",
'Juvenile Nonfiction': "Children's Nonfiction",
'Science': "Nonfiction",
'Poetry': "Fiction"}

books['broad_category']=books['categories'].map(category_mapping)


In [13]:
books[~(books['broad_category'].isna())]

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,broad_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 | A NOVEL THAT READERS and criti...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 | A memorable, mesmerizing heroi...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 | Tricked once more by his wily ...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 | Joe and his parents are enjoyi...,Children's Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014 | A Few Weeks After Marrying An ...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279 | ""Cornell Woolrich's novels def...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 | Rescued from the lockers in wh...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 | This book is the story of a yo...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 | This collection of the timeles...,Nonfiction


In [ ]:
# def classify_book_with_gemini(tagged_description):
#     if not tagged_description or len(str(tagged_description)) < 10:
#         return "Other"

#     # Define your focus categories here (equivalent to fiction_categories in the video)
#     categories = ["Fiction", "Nonfiction"]

#     # The prompt acts as the "Zero-Shot" instruction
#     prompt = f"""
#     You are an expert book librarian. 
#     Classify the following book description into one of these specific categories: {', '.join(categories)}.
    
#     Constraint: Your output must be exactly one of those two words.
    
#     Description: {tagged_description}
#     """
    
#     try:
#         response = llm.invoke(prompt)
#         result = response.content.strip()
        
#         # Double check that Gemini didn't give extra text
#         if "Nonfiction" in result: return "Nonfiction"
#         if "Fiction" in result: return "Fiction"
#         return "Other"
        
#     except Exception as e:
#         print(f"Error: {e}")
#         return "Other"

In [11]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions

# --- SETUP RETRY LOGIC ---
# This tells Python: "If the API fails, wait 2s, then 4s, then 8s... up to 5 times"
@retry(
    stop=stop_after_attempt(5), 
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(Exception) # Retries on any crash (like internet or API limit)
)
def classify_book_with_gemini_robust(tagged_description):
    # 1. Validation: Skip bad data immediately
    if not tagged_description or len(str(tagged_description)) < 5:
        return "Other"

    categories = ["Fiction", "Nonfiction"]

    prompt = f"""
    You are an expert librarian. 
    Classify the following book description into strictly one of these categories: {categories}.
    
    Rules:
    1. Output strictly ONE word. 
    2. Do NOT write sentences.
    3. If you are unsure, choose the closest match.

    Description: {tagged_description}
    """
    
    # 2. Call LLM (Works for LangChain)
    try:
        response = llm.invoke(prompt)
        raw_result = response.content.strip()
        
        # 3. CLEANING & LOGIC FIX
        # Remove punctuation like "Fiction." -> "Fiction"
        clean_result = raw_result.replace(".", "").replace('"', "").strip()
        
        # Check "Nonfiction" FIRST to avoid the substring bug
        if "Nonfiction" in clean_result:
            return "Nonfiction"
        elif "Fiction" in clean_result:
            return "Fiction"
        else:
            # Fallback if Gemini says something weird like "Bio-graphy"
            return "Other"
            
    except Exception as e:
        print(f"API Error on book: {e}")
        raise e # We raise the error so 'tenacity' knows to retry!

In [23]:
# ... (Keep your existing imports and classify_book_with_gemini_robust function here) ...

print("--- 🔍 STARTING SANITY CHECK ---")

# 1. Define a clear 'Fiction' description (Harry Potter style)
test_fiction = "A young wizard discovers his magical heritage and attends a school of witchcraft to fight dark forces."

# 2. Define a clear 'Nonfiction' description (History style)
test_nonfiction = "A detailed history of the Roman Empire, exploring the political and social causes of its decline."

# 3. Run your function
result_1 = classify_book_with_gemini_robust(test_fiction)
result_2 = classify_book_with_gemini_robust(test_nonfiction)

# 4. Print results (This is your 'Check')
print(f"Test 1 (Should be Fiction): {result_1}")
print(f"Test 2 (Should be Nonfiction): {result_2}")

if result_1 == "Fiction" and result_2 == "Nonfiction":
    print("✅ System is working correctly!")
else:
    print("❌ Something is wrong. Check your prompt.")

--- 🔍 STARTING SANITY CHECK ---
Test 1 (Should be Fiction): Fiction
Test 2 (Should be Nonfiction): Nonfiction
✅ System is working correctly!


In [24]:
# A small dataset of 'Known' books
test_cases = [
    ("The Lord of the Rings is an epic high-fantasy novel by J.R.R. Tolkien.", "Fiction"),
    ("Sapiens: A Brief History of Humankind explores the biology and history of humans.", "Nonfiction"),
    ("Becoming is the memoir of former United States First Lady Michelle Obama.", "Nonfiction"),
    ("Dune is a science fiction novel set on the desert planet Arrakis.", "Fiction"),
    ("The Elements of Style is a style guide for writing American English.", "Nonfiction")
]

correct_count = 0

print("\n--- 📊 STARTING BATCH ACCURACY CHECK ---")

for description, expected_label in test_cases:
    # Run your function
    prediction = classify_book_with_gemini_robust(description)
    
    # Check if it matches
    if prediction == expected_label:
        print(f"✅ Correct: {expected_label}")
        correct_count += 1
    else:
        print(f"❌ Wrong! Expected {expected_label}, got {prediction}")
        print(f"   Input: {description[:50]}...")

# Calculate Score
accuracy = (correct_count / len(test_cases)) * 100
print(f"\nFinal Accuracy: {accuracy}%")


--- 📊 STARTING BATCH ACCURACY CHECK ---
✅ Correct: Fiction
✅ Correct: Nonfiction
✅ Correct: Nonfiction
✅ Correct: Fiction
✅ Correct: Nonfiction

Final Accuracy: 100.0%


In [ ]:
import random

print("--- 🎲 RUNNING MINI-BATCH TEST (10 Books) ---")

# 1. Get 5 random Fiction books
fiction_samples = books[books['broad_category'] == 'Fiction'].sample(5)['tagged_description'].tolist()

# 2. Get 5 random Nonfiction books
nonfiction_samples = books[books['broad_category'] == 'Nonfiction'].sample(5)['tagged_description'].tolist()

# 3. Test Fiction
print("\n--- Testing Fiction Samples ---")
for desc in fiction_samples:
    prediction = classify_book_with_gemini_robust(desc)
    print(f"Prediction: {prediction} | (Expected: Fiction)")

# 4. Test Nonfiction
print("\n--- Testing Nonfiction Samples ---")
for desc in nonfiction_samples:
    prediction = classify_book_with_gemini_robust(desc)
    print(f"Prediction: {prediction} | (Expected: Nonfiction)")